# Screening a library

This tutorial aligns a reference molecule against a small library, first with `MoleculePairBatch` (in memory) and then with `ProfileStore` + `screen` (on disk, streamed shard by shard, which scales to large libraries). Everything here runs on the CPU.

In [1]:
import tempfile
from pathlib import Path

from shepherd_score.conformer_generation import embed_conformer_from_smiles
from shepherd_score.container import Molecule, MoleculePair, MoleculePairBatch
from shepherd_score.screen import ProfileStore, screen

# Reference molecule and a small library
ref = Molecule(embed_conformer_from_smiles('Oc1ccc(CC=C)cc1', MMFF_optimize=True))
smiles = {
    'anisole-allyl': 'COc1ccc(CC=C)cc1',
    'ethylphenol': 'Oc1ccc(CC)cc1',
    'aniline-allyl': 'Nc1ccc(CC=C)cc1',
    'toluene': 'Cc1ccccc1',
    'cyclohexanol': 'OC1CCCCC1',
}
library = {k: Molecule(embed_conformer_from_smiles(s, MMFF_optimize=True)) for k, s in smiles.items()}
len(library)

5

## Batched alignment

`MoleculePairBatch` aligns many pairs at once. Unlike `MoleculePair.align`, which returns an `AlignmentResult`, the batch `align` also stores the score and transform on each pair (`pair.sim_aligned_vol_noH`, `pair.transform_vol_noH`). The `numba` backend runs on the CPU; on a GPU, omit `backend` or use `"triton"`.

In [2]:
pairs = [MoleculePair(ref, mol) for mol in library.values()]
batch = MoleculePairBatch(pairs)
batch.align("vol", backend="numba")

for name, pair in zip(library, pairs):
    print(f"{name:15s} {float(pair.sim_aligned_vol_noH):.3f}")
print(pairs[0].transform_vol_noH.shape)  # (4, 4) transform of the fit molecule onto the reference

anisole-allyl   0.922
ethylphenol     0.956
aniline-allyl   1.000
toluene         0.814
cyclohexanol    0.789
(4, 4)


## Building a profile store

For libraries too large to hold in memory, write the profiles once to a `ProfileStore`. A store only keeps the arrays needed by the `modes` it is created for. `num_surf_points=0` means no surface is stored, which is enough for `"vol"`.

In [3]:
tmp = tempfile.TemporaryDirectory()
vol_path = str(Path(tmp.name) / "library_vol")

with ProfileStore.create(vol_path, num_surf_points=0, modes=("vol",)) as store:
    for name, mol in library.items():
        store.add(mol, id=name)

store = ProfileStore.open(vol_path)
len(store), store.modes

(5, ('vol',))

## Screening

`screen` aligns every molecule in the store to the query and returns the best `top_k` hits, best first. Each `Hit` is a named tuple with `id`, `score` and `transform`. (The `vol` mode ignores element identity, so the aniline analog, which differs from the reference only by N vs O, scores 1.0.)

In [4]:
hits = screen(ref, store, mode="vol", top_k=3, backend="numba")
for h in hits:
    print(f"{h.id:15s} {h.score:.3f}")
hits[0].transform.shape

aniline-allyl   1.000
ethylphenol     0.956
anisole-allyl   0.923


(4, 4)

## Storing several modes

A store can hold the arrays for more than one mode. Surface modes need `num_surf_points` (the query must use the same number) and partial charges; here `charge_model="mmff"` avoids requiring xTB (the default is `charge_model='xtb_water'`). The `vol_color` mode adds pharmacophore-like color features to the volumetric score.

In [5]:
n_surf = 100
ref_s = Molecule(ref.mol, num_surf_points=n_surf, charge_model="mmff", pharm_multi_vector=False)

multi_path = str(Path(tmp.name) / "library_multi")
with ProfileStore.create(multi_path, num_surf_points=n_surf, modes=("vol", "vol_color", "surf_esp")) as store:
    for name, mol in library.items():
        store.add(Molecule(mol.mol, num_surf_points=n_surf, charge_model="mmff", pharm_multi_vector=False), id=name)

multi = ProfileStore.open(multi_path)
print(multi.modes)
for mode in ("vol_color", "surf_esp"):
    hits = screen(ref_s, multi, mode=mode, top_k=3, backend="numba")
    print(mode, [(h.id, round(float(h.score), 3)) for h in hits])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


('vol', 'vol_color', 'surf_esp')


vol_color [('aniline-allyl', 0.929), ('ethylphenol', 0.905), ('anisole-allyl', 0.768)]


surf_esp [('aniline-allyl', 0.906), ('ethylphenol', 0.864), ('anisole-allyl', 0.806)]


## Going further

- **GPU**: omit `backend` (it defaults to `"triton"` when CUDA is available) or pass `backend="triton"`; use `ndev=N` to spread a screen over several GPUs and `cpu_threads=` to cap CPU threads on the CPU backend (also accepted by `MoleculePairBatch`).
- **More modes** and what each store must contain: see the modes guide (`modes.md`).
- **Batched/accelerated API** reference, including `screen_many` for several queries: see `accelerated_api.md`.